In [1]:
LAST_NAME = "Gabilan"
STUDENT_ID = "TUPM-26-2636"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(digit) for digit in STUDENT_ID if digit.isdigit())
NAME_LENGTH = len(LAST_NAME)

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR = (SEED_DIGIT % 5) + 1

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Target Operating System:", TARGET_OS)
print("Target Behavior Class:", TARGET_BEHAVIOR)

Student: Gabilan
Student ID: TUPM-26-2636
Seed Digit: 6
ID Sum: 25
Name Length: 7
Target Operating System: Android
Target Behavior Class: 2


In [2]:
import csv
import json
import os
import shutil

In [3]:
INPUT_FILE = "user_behavior_dataset.csv"

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILTERED_FILE = os.path.join(OUTPUT_DIR, "filtered_mobile_users.csv")
SUMMARY_FILE = os.path.join(OUTPUT_DIR, "mobile_usage_summary.json")
BACKUP_FILE = os.path.join(OUTPUT_DIR, "mobile_users_backup.csv")

print("Input File:", INPUT_FILE)
print("Output Directory:", OUTPUT_DIR)

Input File: user_behavior_dataset.csv
Output Directory: output


In [4]:
total_records = 0
os_records = 0
target_records = []
invalid_records = 0

with open(INPUT_FILE, "r", newline="", encoding="utf-8-sig") as file:
    reader = csv.DictReader(file)

    for row in reader:
        total_records += 1

        try:
            operating_system = row["Operating System"].strip()

            if operating_system != TARGET_OS:
                continue

            os_records += 1

            behavior = int(row["User Behavior Class"])
            battery = float(row["Battery Drain (mAh/day)"])
            data_usage = float(row["Data Usage (MB/day)"])

            if behavior != TARGET_BEHAVIOR:
                continue

            if battery < 0 or data_usage < 0:
                invalid_records += 1
                continue

            target_records.append({
                "User ID": row["User ID"],
                "Device Model": row["Device Model"],
                "Operating System": operating_system,
                "Battery Drain (mAh/day)": battery,
                "Data Usage (MB/day)": data_usage
            })

        except (KeyError, ValueError, TypeError, AttributeError):
            invalid_records += 1

print("Total Records:", total_records)
print("Records Matching OS:", os_records)
print("Records in Target Behavior Class:", len(target_records))
print("Invalid Records:", invalid_records)

Total Records: 700
Records Matching OS: 554
Records in Target Behavior Class: 114
Invalid Records: 0


In [5]:
if target_records:
    average_battery = sum(
        record["Battery Drain (mAh/day)"]
        for record in target_records
    ) / len(target_records)

    average_data = sum(
        record["Data Usage (MB/day)"]
        for record in target_records
    ) / len(target_records)

else:
    average_battery = None
    average_data = None

print("Average Battery Drain:", average_battery)
print("Average Data Usage:", average_data)

Average Battery Drain: 887.1140350877193
Average Data Usage: 451.43859649122805


In [6]:
fieldnames = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

with open(FILTERED_FILE, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()
    writer.writerows(target_records)

print("Filtered CSV saved:", FILTERED_FILE)

Filtered CSV saved: output/filtered_mobile_users.csv


In [7]:
summary = {
    "Student": LAST_NAME,
    "Student ID": STUDENT_ID,
    "Operating System": TARGET_OS,
    "User Behavior Class": TARGET_BEHAVIOR,
    "Total Records in Dataset": total_records,
    "Records Matching Operating System": os_records,
    "Records in Target Behavior Class": len(target_records),
    "Average Battery Drain (mAh/day)": average_battery,
    "Average Data Usage (MB/day)": average_data,
    "Invalid Records": invalid_records
}

with open(SUMMARY_FILE, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON Summary saved:", SUMMARY_FILE)

JSON Summary saved: output/mobile_usage_summary.json


In [8]:
shutil.copy2(FILTERED_FILE, BACKUP_FILE)

with open(FILTERED_FILE, "r", newline="", encoding="utf-8") as file:
    exported_count = sum(1 for row in csv.DictReader(file))

count_verified = exported_count == len(target_records)

print("Selected Records:", len(target_records))
print("Exported Records:", exported_count)
print("Count Verification:", count_verified)
print("Backup Created:", os.path.exists(BACKUP_FILE))

if os.path.exists(BACKUP_FILE):
    print("Backup File Size:", os.path.getsize(BACKUP_FILE), "bytes")

Selected Records: 114
Exported Records: 114
Count Verification: True
Backup Created: True
Backup File Size: 4573 bytes


In [9]:
print("===== MOBILE DEVICE USAGE REPORT =====")
print("Operating System:", TARGET_OS)
print("Behavior Class:", TARGET_BEHAVIOR)
print("Total Dataset Records:", total_records)
print("Matching OS Records:", os_records)
print("Target Class Records:", len(target_records))
print("Average Battery Drain:", average_battery)
print("Average Data Usage:", average_data)
print("Count Verification:", count_verified)
print("Filtered CSV:", FILTERED_FILE)
print("JSON Summary:", SUMMARY_FILE)
print("Backup File:", BACKUP_FILE)
print("======================================")

===== MOBILE DEVICE USAGE REPORT =====
Operating System: Android
Behavior Class: 2
Total Dataset Records: 700
Matching OS Records: 554
Target Class Records: 114
Average Battery Drain: 887.1140350877193
Average Data Usage: 451.43859649122805
Count Verification: True
Filtered CSV: output/filtered_mobile_users.csv
JSON Summary: output/mobile_usage_summary.json
Backup File: output/mobile_users_backup.csv
